# Exp No 2: Simple morphing interpolation

In [ ]:
import os

!pip install ninja imageio imageio-ffmpeg opencv-python-headless -q

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(repo_dir := "/content/gh0st-in-the-l00p"):
    !git -C {repo_dir} pull
else:
    !git clone https://github.com/jasonr2048/gh0st-in-the-l00p.git {repo_dir}

%cd {repo_dir}

In [ ]:
import json

drive_root    = "/content/drive/MyDrive/Gh0st in the Loop"
prepared_root = f"{drive_root}/dataset/prepared"
output_dir    = f"{drive_root}/outputs"
output_path   = f"{output_dir}/interpolation_flow.mp4"

sequence_path = f"{repo_dir}/data/sequence.json"

with open(sequence_path) as f:
    sequence = json.load(f)

print(f"Loaded sequence: {sequence['total_images']} images across {len(sequence['sets'])} sets")
for s in sequence["sets"]:
    print(f"  {s['name']}: {len(s['images'])} images")
if sequence.get("undetected"):
    print(f"  ({len(sequence['undetected'])} images had no keypoints detected)")

# ── Timing ────────────────────────────────────────────────────────────────────
fps              = 24
hold_frames      = 12   # frames to hold on each image (0 = no hold)
morph_frames     = 24   # frames for morph between images within a set
between_frames   = 48   # frames for morph between last image of one set and first of next

# ── Output ────────────────────────────────────────────────────────────────────
resolution       = (1080, 1920)  # width, height

In [ ]:
from PIL import Image, ImageOps

print("Selected sets:")
for s in sequence["sets"]:
    images = [ImageOps.exif_transpose(Image.open(p).convert("RGB")) for p in s["images"]]
    grid = Image.new("RGB", (len(images) * 200, 200))
    for i, img in enumerate(images):
        grid.paste(img.resize((200, 200)), (i * 200, 0))
    print(f"\n{s['name']} ({len(images)} images):")
    display(grid)

In [ ]:
import cv2
import numpy as np

def to_cv2(pil_img, size):
    return cv2.resize(np.array(pil_img.convert("RGB")), size)

def to_pil(cv2_img):
    return Image.fromarray(cv2_img)

def optical_flow_morph(img_a, img_b, steps, size):
    a = to_cv2(img_a, size)
    b = to_cv2(img_b, size)

    a_gray = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    b_gray = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        a_gray, b_gray,
        None, 0.5, 3, 15, 3, 5, 1.2, 0
    )

    h, w = a.shape[:2]
    base_x = np.tile(np.arange(w), (h, 1)).astype(np.float32)
    base_y = np.tile(np.arange(h), (w, 1)).T.astype(np.float32)

    frames = []
    for t in np.linspace(0, 1, steps):
        warp_x = (base_x + flow[..., 0] * t).astype(np.float32)
        warp_y = (base_y + flow[..., 1] * t).astype(np.float32)
        warped = cv2.remap(a, warp_x, warp_y, cv2.INTER_LINEAR)
        blended = cv2.addWeighted(warped, 1 - t, b, t, 0)
        frames.append(to_pil(blended))

    return frames

def hold(img, n_frames, size):
    """Repeat a single image for n_frames."""
    frame = to_cv2(img, size).copy()
    return [to_pil(frame)] * n_frames

In [ ]:
size = resolution  # (width, height) for cv2

all_frames = []
all_images = []  # flat list of (image, set_name) in optimised order

for s in sequence["sets"]:
    for img_path in s["images"]:
        img = ImageOps.exif_transpose(Image.open(img_path).convert("RGB"))
        all_images.append((img, s["name"]))

print(f"Total images in sequence: {len(all_images)}")

for i, (img, name) in enumerate(all_images):
    next_img, next_name = all_images[(i + 1) % len(all_images)]

    # Hold on current image
    if hold_frames > 0:
        all_frames.extend(hold(img, hold_frames, size))

    # Determine morph length
    current_set_last = (i == len(all_images) - 1) or (all_images[i + 1][1] != name)
    n_morph = between_frames if current_set_last else morph_frames

    # Morph to next
    all_frames.extend(optical_flow_morph(img, next_img, n_morph, size))

total_seconds = len(all_frames) / fps
print(f"Total frames: {len(all_frames)} ({total_seconds:.1f}s at {fps}fps)")

In [ ]:
preview_indices = np.linspace(0, len(all_frames) - 1, 10, dtype=int)
grid = Image.new("RGB", (10 * 108, 192))
for j, idx in enumerate(preview_indices):
    thumb = all_frames[idx].resize((108, 192))
    grid.paste(thumb, (j * 108, 0))
display(grid)
print("Preview: 10 evenly spaced frames from the full sequence")

In [ ]:
import imageio

os.makedirs(output_dir, exist_ok=True)

with imageio.get_writer(output_path, fps=fps) as writer:
    for i, frame in enumerate(all_frames):
        writer.append_data(np.array(frame))
        if i % 50 == 0:
            print(f"  {i}/{len(all_frames)} frames written...", end="\r")

print(f"\n✅ Video saved to {output_path}")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

data_url = "data:video/mp4;base64," + b64encode(open(output_path, "rb").read()).decode()
display(HTML(f'<video width=540 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))